# 02 · How precise is the corpus, and what does the relevance rule throw away? (RQ4)

`src/refine.py` cites this notebook for the claim that the raw keyword query was only about 47% precise. The
original 150-paper check was not in the shared archive, so this notebook re-does the measurement on fresh,
reproducible samples and extends it to the final corpus.

**Samples** (all drawn with `random_state=42` from the candidate set just before the relevance rule):

| Sample | From | n |
|---|---|---|
| `final_corpus` | the 10,208 papers kept by the rule | 150 |
| `removed_by_rule` | the 7,725 papers removed by the rule | 100 |
| `medicine_extra` | more Medicine papers from the kept set, so that Medicine has 50 papers | 34 |

**Labelling rule.** A paper is **yes** when AI (machine learning, LLMs, learning analytics, tutoring systems,
educational data mining, or education *about* AI) is studied **in an education context**: teaching, learning,
students, teachers, curricula, assessment or educational institutions, medical education included.
It is **no** when education is incidental (patient education, a "school" building, a "teacher-student" network,
students only as survey respondents). **Unsure** when the title and abstract do not decide it.

**Who labelled.** First labels by Claude (AI), from title and abstract, with a one-line reason for every no and
unsure. Two team members should check them: fill `label_member_a` and `label_member_b` in
`data/validation/relevance_labels.csv` (yes / no / unsure). When both are filled, the notebook uses their
consensus instead of the AI label and reports agreement. Precision is reported *strict* (unsure = no) and
*lenient* (unsure = yes).

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
VALID = ROOT / "data" / "validation"
sys.path.insert(0, str(ROOT / "src"))
import clean as P1          # Person 1's cleaning code, used unchanged
SEED = 42
print(ROOT)

/home/claude/work_cc


## 1. Rebuild the candidate set (before the relevance rule)

Person 1's `clean.clean()` is run with the relevance rule switched off. Everything else (duplicate
authorships, version merging, document types) is exactly as in the pipeline.

In [2]:
works = pd.read_csv(PROCESSED / "works.csv")
auths = pd.read_csv(PROCESSED / "authorships.csv")
rule = P1.is_relevant
P1.is_relevant = lambda title, abstract: True       # rule off
cand, _, _, log = P1.clean(works, auths)
P1.is_relevant = rule                                # rule back on
final_ids = set(pd.read_csv(PROCESSED / "works_clean.csv", usecols=["work_id"])["work_id"])
cand["kept_by_rule"] = cand["work_id"].isin(final_ids)

n_cand, n_kept = len(cand), int(cand["kept_by_rule"].sum())
n_removed = n_cand - n_kept
assert (n_cand, n_kept) == (17933, 10208), (n_cand, n_kept)
print(f"candidates {n_cand:,} | kept {n_kept:,} | removed {n_removed:,}")

candidates 17,933 | kept 10,208 | removed 7,725


## 2. Draw the samples and check they match the labelled file

In [3]:
k = cand[cand["kept_by_rule"]].sample(150, random_state=SEED).assign(sample="final_corpus")
r = cand[~cand["kept_by_rule"]].sample(100, random_state=SEED).assign(sample="removed_by_rule")
pool = cand[cand["kept_by_rule"] & (cand["field"] == "Medicine") & ~cand["work_id"].isin(k["work_id"])]
m = pool.sample(50 - int((k["field"] == "Medicine").sum()), random_state=SEED).assign(sample="medicine_extra")
drawn = pd.concat([k, r, m])

labels = pd.read_csv(VALID / "relevance_labels.csv")
assert list(drawn["work_id"]) == list(labels["work_id"]), "the sample no longer matches the labelled file"
print(labels["sample"].value_counts().to_dict())

{'final_corpus': 150, 'removed_by_rule': 100, 'medicine_extra': 34}


## 3. Final label: team consensus if available, otherwise the AI label

In [4]:
for c in ["label_member_a", "label_member_b"]:
    if c not in labels:
        labels[c] = np.nan
team = labels["label_member_a"].notna() & labels["label_member_b"].notna()
agree = team & (labels["label_member_a"] == labels["label_member_b"])
labels["final_label"] = np.where(agree, labels["label_member_a"], labels["label"])
labels["final_source"] = np.where(agree, "team", np.where(team, "AI (team disagree: resolve)", "AI"))
if team.any():
    print(f"team-labelled: {team.sum()} | member agreement: {agree.sum() / team.sum():.0%} | "
          f"agreement with AI: {(labels.loc[agree, 'label_member_a'] == labels.loc[agree, 'label']).mean():.0%}")
else:
    print("No team labels yet: every result below rests on the AI labels.")
labels["final_source"].value_counts()

No team labels yet: every result below rests on the AI labels.


final_source
AI    284
Name: count, dtype: int64

## 4. Results

In [5]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def share(df):
    n = len(df)
    y = int((df["final_label"] == "yes").sum())
    u = int((df["final_label"] == "unsure").sum())
    lo, hi = wilson(y, n)
    return {"n": n, "yes": y, "unsure": u, "no": n - y - u,
            "strict": y / n, "strict_95ci": f"{lo:.0%}-{hi:.0%}", "lenient": (y + u) / n}

fin = labels[labels["sample"] == "final_corpus"]
rem = labels[labels["sample"] == "removed_by_rule"]
med = labels[labels["sample"].isin(["final_corpus", "medicine_extra"]) & (labels["field"] == "Medicine")]
nonmed = fin[fin["field"] != "Medicine"]

res = pd.DataFrame({
    "Precision of the final corpus": share(fin),
    "  ... Medicine only": share(med),
    "  ... without Medicine": share(nonmed),
    "On-topic share among papers removed by the rule": share(rem),
}).T
res[["strict", "lenient"]] = res[["strict", "lenient"]].astype(float).round(3)
res

,n,yes,unsure,no,strict,strict_95ci,lenient
Precision of the final corpus,150,110,13,27,0.733,66%-80%,0.820
... Medicine only,50,21,3,26,0.420,29%-56%,0.480
... without Medicine,134,102,12,20,0.761,68%-83%,0.851
On-topic share among papers removed by the rule,100,17,15,68,0.170,11%-26%,0.320


In [6]:
# What the samples imply for the whole candidate set (weighted by the sizes of kept / removed)
p_k_s, p_k_l = res.loc["Precision of the final corpus", ["strict", "lenient"]]
p_r_s, p_r_l = res.loc["On-topic share among papers removed by the rule", ["strict", "lenient"]]
est = {}
for tag, pk, pr in [("strict", p_k_s, p_r_s), ("lenient", p_k_l, p_r_l)]:
    on_kept, on_removed = pk * n_kept, pr * n_removed
    est[tag] = {
        "on-topic papers kept": round(on_kept),
        "on-topic papers removed": round(on_removed),
        "precision before the rule (raw query, after cleaning)": round((on_kept + on_removed) / n_cand, 3),
        "recall of the rule (share of on-topic papers kept)": round(on_kept / (on_kept + on_removed), 3),
    }
pd.DataFrame(est)

,strict,lenient
on-topic papers kept,7482.000,8371.000
on-topic papers removed,1313.000,2472.000
"precision before the rule (raw query, after cleaning)",0.490,0.605
recall of the rule (share of on-topic papers kept),0.851,0.772


## 5. Recall check on core journals

Every paper in a core AI-in-education journal is on topic by definition. Among the candidates the query found
in the two largest core journals, how many does the relevance rule keep?

In [7]:
core = ["Computers and Education Artificial Intelligence",
        "International Journal of Artificial Intelligence in Education"]
cj = cand[cand["source_name"].isin(core)]
t = cj.groupby("source_name")["kept_by_rule"].agg(found_by_query="size", kept_by_rule="sum")
t.loc["Both"] = t.sum()
t["share_kept"] = (t["kept_by_rule"] / t["found_by_query"]).round(3)
t

,found_by_query,kept_by_rule,share_kept
source_name,,,
Computers and Education Artificial Intelligence,88,83,0.943
International Journal of Artificial Intelligence in Education,56,42,0.750
Both,144,125,0.868


In [8]:
# Core-journal papers the rule removed: which words did they use instead of ours?
cj.loc[~cj["kept_by_rule"], ["source_name", "year", "title"]].head(20)

,source_name,year,title
780,International Journal of Artificial Intelligen...,2020,Measuring the Quality of Test-based Exercises ...
884,International Journal of Artificial Intelligen...,2020,Intelligent Feedback on Hypothesis Testing
907,International Journal of Artificial Intelligen...,2020,Stat-Knowlab. Assessment and Learning of Stati...
1325,Computers and Education Artificial Intelligence,2021,Decision tree learning through a Predictive Mo...
1423,International Journal of Artificial Intelligen...,2021,Linking Dialogue with Student Modelling to Cre...
1734,International Journal of Artificial Intelligen...,2021,Knowledge-Based Design Analytics for Authoring...
2025,International Journal of Artificial Intelligen...,2021,Early prediction of writing quality using keys...
2545,Computers and Education Artificial Intelligence,2022,The development and implementation of a comput...
3016,International Journal of Artificial Intelligen...,2022,Estimating a Dose-Response Relationship in Qua...
3987,Computers and Education Artificial Intelligence,2023,"Automated coding of student chats, a trans-top..."


**Recall of the query itself** needs the total number of EU papers in these journals, which only the API can
give. The cell below asks OpenAlex for it; it needs internet access and is skipped when offline.

In [9]:
import urllib.request
EU = "AT|BE|BG|HR|CY|CZ|DK|EE|FI|FR|DE|GR|HU|IE|IT|LV|LT|LU|MT|NL|PL|PT|RO|SK|SI|ES|SE"
sources = {"Computers and Education Artificial Intelligence": "S4210183364",
           "International Journal of Artificial Intelligence in Education": "S171267539"}
raw = pd.read_csv(PROCESSED / "works.csv", usecols=["source_id"])
rows = []
for name, sid in sources.items():
    found = int((raw["source_id"] == f"https://openalex.org/{sid}").sum())
    url = (f"https://api.openalex.org/works?filter=primary_location.source.id:{sid},"
           f"authorships.institutions.country_code:{EU},publication_year:2020-2025&per-page=1")
    try:
        total = json.load(urllib.request.urlopen(url, timeout=20))["meta"]["count"]
    except Exception as e:
        total = np.nan
        print(f"offline or blocked ({type(e).__name__}): run this cell with internet access")
    rows.append({"journal": name, "EU papers in OpenAlex": total, "found by our query": found,
                 "query recall": round(found / total, 3) if total == total else np.nan})
pd.DataFrame(rows)

offline or blocked (URLError): run this cell with internet access
offline or blocked (URLError): run this cell with internet access


,journal,EU papers in OpenAlex,found by our query,query recall
0,Computers and Education Artificial Intelligence,NaN,89,NaN
1,International Journal of Artificial Intelligen...,NaN,58,NaN


## 6. Examples of false hits in the final corpus

In [10]:
fin.loc[fin["final_label"] != "yes", ["field", "title", "final_label", "note"]].sort_values("field")

,field,title,final_label,note
74,Arts and Humanities,Projekt Enrich Europeana+ in Državni arhiv v Z...,no,AI and citizen science in archives
5,Arts and Humanities,Prayer and AI: Exploring the Impact on Orthodo...,unsure,religious practice of high-school pupils and A...
108,Arts and Humanities,Gli studi linguistici,no,history of linguistic studies at a school
122,Arts and Humanities,"DIGITAL DOPING FOR HISTORIANS: CAN HISTORY, ME...",unsure,AI and historical theory; classroom mentioned ...
14,Arts and Humanities,OVERCOMING THE CREATIVE CRISIS OF THE FASHION ...,no,generative AI in fashion design
106,"Business, Management and Accounting",Transitions into the futures: AI disruption an...,unsure,AI and employability of hospitality graduates
134,Computer Science,ASCDomain: Domain Invariant Device-Adversarial...,no,acoustic scene classification (knowledge disti...
7,Computer Science,Natural Language Processing and Large Language...,no,NLP textbook chapter
101,Computer Science,Can Generative Artificial Intelligence Write a...,no,generative AI writing journal articles; resear...
89,Computer Science,Are digital natives ready to work in a digital...,unsure,students' readiness for digital workplaces; AI...


## 7. What this shows

*Numbers in this section are filled from the outputs above (AI labels, before the team check).*

- **Precision of the final corpus: 73% strict (95% CI 66–80%), 82% lenient.** About one paper in four to five is
  not really about AI in education.
- **Medicine is the weak spot: 42% strict (n = 50, 95% CI 29–56%), against 76% for the rest of the corpus.** Most false hits are
  clinical AI papers where "education" means patient education or the paper only mentions training. This
  confirms the warning in the report: the health cluster of RQ3 is partly off-topic medicine.
- **The rule removes mostly noise:** 17% of removed papers are on topic (32% lenient).
- **Weighted back to the whole candidate set,** the raw query (after cleaning) was about 49% precise. This is
  consistent with the ~47% that `refine.py` cites. The rule keeps about 85% of the on-topic papers (77%
  lenient): it trades some recall for a large gain in precision.
- **Core journals:** the rule keeps 94% of the *Computers and Education: AI* papers but only 75% of the *IJAIED*
  papers. The removed IJAIED papers use words that are not in our AI list: "adaptive tutoring", "knowledge
  tracing", "essay scoring", "recommendation system", "student modelling".

**What this does not show.** Papers the query never found (other languages, other vocabulary) are invisible
here. Only the API check in section 5 measures that, for two journals.

**What could overturn it.** The AI labels are a first pass. If the team's labels disagree often, especially on
the unsure cases, the figures above change. Fill the two member columns and re-run the notebook.